# GPT-Neo residual stream — cached to HDF5

Same run as the in-memory example, but with `cache_outputs=True` so
activations stream to an HDF5 file instead of staying in RAM. Downloads on
first run: WikiText-2 (~5 MB) and GPT-Neo 125M weights (~500 MB).

In [ ]:
import sys
from pathlib import Path

# Make the repo-local examples._utils package importable when this notebook
# is opened directly from examples/text/, without installing anything extra.
sys.path.insert(0, str(Path.cwd().parents[1]))

from transformers import AutoModelForCausalLM, AutoTokenizer

from examples._utils.data import activation_loader
from examples._utils.text import WikiTextSamples
from nnact import ActivationPipeline
from nnact._model._hooked import HookedModel

MODEL = "EleutherAI/gpt-neo-125m"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token  # GPT-Neo ships without a pad token
model = AutoModelForCausalLM.from_pretrained(MODEL)  # has .logits, real next-token head

In [ ]:
dataset = WikiTextSamples(tokenizer, n=512, max_length=64)
hooked = HookedModel(model)

# The residual stream: every transformer block, plus the final layer norm.
num_layers = model.config.num_layers
LAYERS = [f"transformer.h.{i}" for i in range(num_layers)] + ["transformer.ln_f"]
print(f"{len(dataset)} passages | {len(LAYERS)} layers: {LAYERS}")

In [ ]:
# cache_outputs=True streams every batch's activations to an HDF5 file at
# cache_dir/activations.h5 instead of accumulating them in memory. Passing a
# tokenizer for a "token" pipeline also attaches token-level metadata:
# token_ids and decoded tokens.
pipeline = ActivationPipeline(
    model,
    LAYERS,
    output_type="token",
    tokenizer=tokenizer,
    cache_outputs=True,
    cache_dir=Path.cwd() / "cache",
)
loader = activation_loader(dataset, batch_size=16)
activations = pipeline.run(loader)
activations.summary()
activations.print_cache_info()

In [ ]:
# Slicing by offsets pulls out one passage's own tokens across every layer.
import numpy as np

offsets = activations.offsets
start, end = int(offsets[0]), int(offsets[1])
sample_activations = {
    name: tensor[start:end] for name, tensor in activations.activations.items()
}
print(
    "first passage ->",
    len(sample_activations),
    "layers,",
    tuple(next(iter(sample_activations.values())).shape),
    "each,",
)

# Residual stream norm grows with depth - the usual GPT-2/GPT-Neo picture.
for name, tensor in sample_activations.items():
    print(f"  {name:<18} mean L2 norm = {np.linalg.norm(tensor, axis=-1).mean():6.2f}")